# 疑似ラベル作成 + 再学習
testデータをモデルで推論→高confのみ疑似ラベルとして追加→再学習

## 0. セットアップ

In [ ]:
import os
os.chdir(r'C:\compe')

import json
import numpy as np
import pandas as pd
import torch
import shutil
import yaml
from PIL import Image as PILImage
from ultralytics import YOLO, RTDETR
from rfdetr import RFDETRBase
from ensemble_boxes import weighted_boxes_fusion
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'なし')

with open(r'C:\compe\train_dataset.json') as f:
    train_data = json.load(f)
with open(r'C:\compe\test_dataset.json') as f:
    test_data = json.load(f)

cat_ids = sorted([c['id'] for c in train_data['categories']])
yolo_to_category = cat_ids
category_to_yolo = {cat_id: i for i, cat_id in enumerate(cat_ids)}
cat_names = {c['id']: c['name'] for c in train_data['categories']}

TEST_DIR  = r'C:\compe\images\test'
TRAIN_DIR = r'C:\compe\images\train'
test_files = sorted(os.listdir(TEST_DIR))

fname_to_info = {img['file_name']: img for img in test_data['images']}
img_id_to_info = {img['id']: img for img in test_data['images']}

print(f'test画像数: {len(test_files)}')

## 1. モデル読み込み

In [ ]:
model_yolom = YOLO(r'C:\compe\runs\full2\weights\best.pt')
print('yolo26m loaded')

model_yolol = YOLO(r'C:\Users\tamkn\Downloads\best.pt')
print('yolo26l loaded')

model_rfdetr = RFDETRBase(
    pretrain_weights=r'C:\compe\runs\rfdetr\checkpoint_best_total.pth',
    num_classes=32
)
model_rfdetr.optimize_for_inference()
print('RF-DETR loaded')

## 2. 推論設定

In [ ]:
# 疑似ラベル用は高confのみ採用
PSEUDO_CONF      = 0.3    # これ以上の信頼度のみ採用
IOU              = 0.6
MAX_DET          = 300
WBF_IOU          = 0.55
WBF_SKIP_BOX_THR = 0.3    # 低confを除外
RFDETR_THR       = 0.3    # RF-DETRも高conf
WEIGHTS          = [0.5, 3.0, 2.0]  # yolo26m : RF-DETR : yolo26l

## 3. test画像を推論して疑似ラベル作成

In [ ]:
def predict_yolo(model, img_path, conf, iou):
    result = model.predict(
        source=img_path, conf=conf, iou=iou,
        max_det=MAX_DET, imgsz=1024,
        verbose=False, half=True,
    )[0]
    if len(result.boxes) == 0:
        return [], [], []
    boxes  = result.boxes.xyxyn.cpu().numpy().tolist()
    scores = result.boxes.conf.cpu().numpy().tolist()
    labels = result.boxes.cls.cpu().numpy().astype(int).tolist()
    boxes  = [[min(max(v, 0.0), 1.0) for v in box] for box in boxes]
    return boxes, scores, labels


def predict_rfdetr(model, img_path, w, h, threshold):
    img = PILImage.open(img_path).convert('RGB')
    result = model.predict(img, threshold=threshold)
    if len(result) == 0:
        return [], [], []
    valid = result.class_id < 32
    if not valid.any():
        return [], [], []
    boxes  = result.xyxy[valid].astype(float)
    boxes[:, [0, 2]] /= w
    boxes[:, [1, 3]] /= h
    boxes  = np.clip(boxes, 0, 1).tolist()
    scores = result.confidence[valid].tolist()
    labels = result.class_id[valid].tolist()
    return boxes, scores, labels


pseudo_annotations = []
pseudo_images = []
ann_id = max(a['id'] for a in train_data['annotations']) + 1
img_id = max(img['id'] for img in train_data['images']) + 1
accepted = 0
skipped  = 0

for fname in tqdm(test_files):
    img_path = os.path.join(TEST_DIR, fname)

    # image_id取得
    info = fname_to_info.get(fname)
    if info is None:
        stem = Path(fname).stem
        for key, val in fname_to_info.items():
            if Path(key).stem == stem:
                info = val
                break
    if info is None:
        skipped += 1
        continue

    w = info['width']
    h = info['height']

    # 各モデルで推論
    boxes_m, scores_m, labels_m = predict_yolo(model_yolom, img_path, PSEUDO_CONF, IOU)
    boxes_l, scores_l, labels_l = predict_yolo(model_yolol, img_path, PSEUDO_CONF, IOU)
    boxes_rf, scores_rf, labels_rf = predict_rfdetr(model_rfdetr, img_path, w, h, RFDETR_THR)

    boxes_list, scores_list, labels_list, weights_used = [], [], [], []
    if len(boxes_m) > 0:
        boxes_list.append(boxes_m);  scores_list.append(scores_m);  labels_list.append(labels_m);  weights_used.append(WEIGHTS[0])
    if len(boxes_rf) > 0:
        boxes_list.append(boxes_rf); scores_list.append(scores_rf); labels_list.append(labels_rf); weights_used.append(WEIGHTS[1])
    if len(boxes_l) > 0:
        boxes_list.append(boxes_l);  scores_list.append(scores_l);  labels_list.append(labels_l);  weights_used.append(WEIGHTS[2])

    if not boxes_list:
        skipped += 1
        continue

    boxes_f, scores_f, labels_f = weighted_boxes_fusion(
        boxes_list, scores_list, labels_list,
        weights=weights_used,
        iou_thr=WBF_IOU,
        skip_box_thr=WBF_SKIP_BOX_THR,
    )

    if len(boxes_f) == 0:
        skipped += 1
        continue

    # 画像情報を追加
    pseudo_images.append({
        'id': img_id,
        'file_name': fname,
        'width': w,
        'height': h,
        'date_captured': info.get('date_captured', ''),
        'license': 0,
        'flickr_url': info.get('flickr_url', ''),
        'coco_url': info.get('coco_url', ''),
    })

    # アノテーションを追加
    for box, score, label in zip(boxes_f, scores_f, labels_f):
        x1, y1, x2, y2 = box
        bx = x1 * w
        by = y1 * h
        bw = (x2 - x1) * w
        bh = (y2 - y1) * h
        pseudo_annotations.append({
            'id': ann_id,
            'image_id': img_id,
            'category_id': yolo_to_category[int(label)],
            'bbox': [bx, by, bw, bh],
            'area': bw * bh,
            'segmentation': [],
            'iscrowd': 0,
        })
        ann_id += 1

    img_id += 1
    accepted += 1

print(f'疑似ラベル採用: {accepted}枚')
print(f'スキップ: {skipped}枚')
print(f'疑似アノテーション数: {len(pseudo_annotations)}')
print(f'1枚あたり平均: {len(pseudo_annotations)/max(accepted,1):.1f}件')

## 4. train_dataset.jsonに疑似ラベルを追加して保存

In [ ]:
# 元のJSONをバックアップ
import shutil
shutil.copy(r'C:\compe\train_dataset.json', r'C:\compe\train_dataset_backup.json')
print('バックアップ完了')

# 疑似ラベルを追加
pseudo_train = {
    'images':      train_data['images'] + pseudo_images,
    'annotations': train_data['annotations'] + pseudo_annotations,
    'categories':  train_data['categories'],
    'info':        train_data.get('info', {}),
    'licenses':    train_data.get('licenses', []),
}

with open(r'C:\compe\train_pseudo.json', 'w') as f:
    json.dump(pseudo_train, f)

print(f'元train画像数: {len(train_data["images"])}')
print(f'疑似ラベル追加後: {len(pseudo_train["images"])}')
print(f'元アノテーション数: {len(train_data["annotations"])}')
print(f'疑似ラベル追加後: {len(pseudo_train["annotations"])}')

## 5. testの画像をtrainフォルダにコピー

In [ ]:
# 採用した疑似ラベル画像をtrainフォルダにシンボリックリンク
pseudo_fnames = {img['file_name'] for img in pseudo_images}

for fname in tqdm(pseudo_fnames):
    src = os.path.join(TEST_DIR, fname)
    dst = os.path.join(TRAIN_DIR, fname)
    if not os.path.exists(dst) and os.path.exists(src):
        try:
            os.symlink(src, dst)
        except:
            shutil.copy(src, dst)

print(f'画像配置完了: {len(pseudo_fnames)}枚')

## 6. YOLOラベルを疑似ラベル分も作成

In [ ]:
# 疑似ラベル分のtxtを作成
pseudo_ann_map = defaultdict(list)
for ann in pseudo_annotations:
    pseudo_ann_map[ann['image_id']].append(ann)

for img in tqdm(pseudo_images):
    stem = Path(img['file_name']).stem
    w, h = img['width'], img['height']
    lines = []
    for ann in pseudo_ann_map[img['id']]:
        cls = category_to_yolo[ann['category_id']]
        x, y, bw, bh = ann['bbox']
        cx = (x + bw / 2) / w
        cy = (y + bh / 2) / h
        bw /= w
        bh /= h
        lines.append(f'{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
    with open(rf'C:\compe\labels\train\{stem}.txt', 'w') as f:
        f.write('\n'.join(lines))

print('YOLOラベル作成完了！')

## 7. 再学習

In [ ]:
from ultralytics import YOLO

# yaml更新（全データ）
dataset_config = {
    'path': r'C:\compe',
    'train': 'images/train',
    'val': 'images/train',
    'names': {i: cat_names[cat_id] for i, cat_id in enumerate(cat_ids)}
}
with open(r'C:\compe\data_pseudo.yaml', 'w') as f:
    yaml.dump(dataset_config, f, allow_unicode=True)

print(f'学習画像数: {len(os.listdir(TRAIN_DIR))}枚')

# yolo26mで再学習（疑似ラベル込み）
model = YOLO(r'C:\compe\runs\full2\weights\best.pt')
model.train(
    data=r'C:\compe\data_pseudo.yaml',
    epochs=15,
    imgsz=1024,
    batch=4,
    hsv_v=0.7,
    val=False,
    project=r'C:\compe\runs',
    name='pseudo',
)
print('疑似ラベル再学習完了！')